In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score 

In [2]:
X_train = pd.read_csv("../data/processed/X_train.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").squeeze()

X_test = pd.read_csv("../data/processed/X_test.csv")
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

Multiple Linear Regression Model

In [6]:
from sklearn.linear_model import LinearRegression

mlr = LinearRegression(
    fit_intercept=True,
    copy_X=True,
    n_jobs=-1,
)

mlr.fit(X_train,y_train)

print("coeffients :",mlr.coef_)
print("intercept :",mlr.intercept_)

y_pred_lr = mlr.predict(X_test)

print("first 10 predictions :",y_pred_lr[:10])
print(f"RMSE score : {root_mean_squared_error(y_test,y_pred_lr):.4f}")
print(f"MAE score : {mean_absolute_error(y_test,y_pred_lr):.4f}")
print(f"R2 : {r2_score(y_test,y_pred_lr):.4f}")

coeffients : [ 7.21210766e-01  3.86312742e-01 -1.08484178e-01 -5.25313301e-02
  2.85127361e-01 -4.51906139e-01  5.28704142e+00  9.02795183e-01
  1.26617830e-01 -3.86986728e-01 -1.44793961e-02 -5.81130255e+00
 -1.75127902e+02 -6.49224985e+00  1.15502124e+00  5.70626997e+00
 -8.46656882e-02]
intercept : 13346.715248789722
first 10 predictions : [546.50057618 499.7209888  383.43373548 366.55174284 291.13137725
 274.63567974 220.00664962 202.22133154 176.94822014 242.83815979]
RMSE score : 31.7792
MAE score : 25.0809
R2 : 0.8763


Gradient Boosting Regressor

In [7]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV

param = {
    'n_estimators' : [200, 300, 400,500],
    'max_depth' : [7,9,11,13],
}

grid = GridSearchCV(
    estimator=GradientBoostingRegressor(random_state=42,learning_rate=0.05),
    param_grid=param,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

grid.fit(X_train,y_train)

y_pred_gb = grid.predict(X_test)

gboost = grid.best_estimator_

print("best parameters :", grid.best_params_)
print(f"RMSE score : {root_mean_squared_error(y_test,y_pred_gb):.4f}")
print(f"MAE score : {mean_absolute_error(y_test,y_pred_gb):.4f}")
print(f"R2 score : {r2_score(y_test,y_pred_gb):.4f}")

best parameters : {'max_depth': 7, 'n_estimators': 200}
RMSE score : 27.9835
MAE score : 22.7132
R2 score : 0.9041


AdaBoost Regressor

In [9]:
from sklearn.ensemble import AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV

base = DecisionTreeRegressor(random_state=42)
adaboost = AdaBoostRegressor(
    estimator=base,
    random_state=42,
    learning_rate=0.05
)

param = {
    'estimator__max_depth' : [2,3,4,5],
    'n_estimators' : [100,200,300,400],
}

grid = GridSearchCV(
    estimator=adaboost,
    param_grid=param,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

grid.fit(X_train,y_train)

y_pred_ada = grid.predict(X_test)

adaboost = grid.best_estimator_

print("best parameters :",grid.best_params_)
print("first 10 predictions :",y_pred_ada[:10])
print(f"RMSE score : {root_mean_squared_error(y_test,y_pred_ada):.4f}")
print(f"MAE score : {mean_absolute_error(y_test,y_pred_ada):.4f}")
print(f"R2 score : {r2_score(y_test,y_pred_ada):.4f}")

best parameters : {'estimator__max_depth': 5, 'n_estimators': 300}
first 10 predictions : [527.2        469.54347826 386.81203008 385.36936937 333.65217391
 331.33628319 270.99052133 270.89361702 213.36774194 273.35810811]
RMSE score : 31.0149
MAE score : 26.8069
R2 score : 0.8822


Random Forest Regressor

In [13]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

param = {
    'n_estimators' : [100,200,300,400],
    'max_depth' : [7,9,13,15],
}

grid = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param,
    cv=5,
    n_jobs=-1,
    scoring='neg_mean_squared_error'
)

grid.fit(X_train,y_train)

rf_model = grid.best_estimator_

y_pred_rf = grid.predict(X_test)

print("best parameters :",grid.best_params_)
print("first 10 predictions :",y_pred_rf[:10])
print(f"RMSE score : {root_mean_squared_error(y_test,y_pred_rf):.4f}")
print(f"MAE score : {mean_absolute_error(y_test,y_pred_rf):.4f}")
print(f"R2 score : {r2_score(y_test,y_pred_rf):.4f}")

best parameters : {'max_depth': 15, 'n_estimators': 200}
first 10 predictions : [504.27       474.78023039 389.61701821 387.07250585 345.86271562
 331.53719516 260.45326754 259.31978366 219.48614607 268.7886735 ]
RMSE score : 27.3561
MAE score : 22.7318
R2 score : 0.9084


CatBoost Regressor

In [16]:
from catboost import CatBoostRegressor
from sklearn.model_selection import GridSearchCV

param = {
    'depth' : [7,9,11,13]
}

grid = GridSearchCV(
    estimator=CatBoostRegressor(iterations=400,loss_function='RMSE',random_seed=42,learning_rate=0.05),
    param_grid=param,
    cv=5,
    n_jobs=-1,
    scoring='neg_mean_squared_error'
)

grid.fit(X_train,y_train)

catboost_model = grid.best_estimator_

y_pred_cat = grid.predict(X_test)

print("best parameters :",grid.best_params_)
print("first 10 predictions :",y_pred_cat[:10])
print(f"RMSE score : {root_mean_squared_error(y_test,y_pred_cat):.4f}")
print(f"MAE score : {mean_absolute_error(y_test,y_pred_cat):.4f}")
print(f"R2 score : {r2_score(y_test,y_pred_cat):.4f}")

0:	learn: 115.2100305	total: 3.2ms	remaining: 1.27s
1:	learn: 111.3349874	total: 6.4ms	remaining: 1.27s
2:	learn: 107.1517396	total: 9.25ms	remaining: 1.22s
3:	learn: 103.3021241	total: 12.2ms	remaining: 1.21s
4:	learn: 99.6957112	total: 15.2ms	remaining: 1.2s
5:	learn: 96.2960603	total: 18.3ms	remaining: 1.2s
6:	learn: 93.0871876	total: 21.4ms	remaining: 1.2s
7:	learn: 89.6688095	total: 24.9ms	remaining: 1.22s
8:	learn: 86.5203198	total: 28.2ms	remaining: 1.23s
9:	learn: 83.6513871	total: 31.2ms	remaining: 1.22s
10:	learn: 80.8886377	total: 34.5ms	remaining: 1.22s
11:	learn: 78.0998441	total: 38.9ms	remaining: 1.26s
12:	learn: 75.4958730	total: 42.1ms	remaining: 1.25s
13:	learn: 73.1915659	total: 45.5ms	remaining: 1.25s
14:	learn: 71.1704833	total: 48.6ms	remaining: 1.25s
15:	learn: 68.9968675	total: 51.8ms	remaining: 1.24s
16:	learn: 67.0155497	total: 55.2ms	remaining: 1.24s
17:	learn: 65.0294343	total: 58.2ms	remaining: 1.24s
18:	learn: 63.1209674	total: 61.2ms	remaining: 1.23s
19:	

In [17]:
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV

model = XGBRegressor(
    learning_rate=0.05,
    random_state=42
)

param = {
    'n_estimators' : [200,300,400,500],
    'max_depth' : [7,9,11,13,15]
}

grid = GridSearchCV(
    param_grid = param,
    estimator = model,
    cv = 5,
    scoring = 'neg_mean_squared_error',
    n_jobs = -1
)

grid.fit(X_train,y_train)

xgboost_model = grid.best_estimator_

print("Best parameters : ",grid.best_params_)

y_pred_xgb = grid.predict(X_test)
print("first 10 predictions :",y_pred_xgb[:10])
print(f"RMSE score : {root_mean_squared_error(y_test,y_pred_xgb):.4f}")
print(f"MAE score : {mean_absolute_error(y_test,y_pred_xgb):.4f}")
print(f"R2 score : {r2_score(y_test,y_pred_xgb):.4f}")

Best parameters :  {'max_depth': 7, 'n_estimators': 200}
first 10 predictions : [488.9496  451.93854 384.66144 396.34802 332.9854  323.0012  272.77658
 262.47763 228.52417 269.06116]
RMSE score : 29.0513
MAE score : 24.3455
R2 score : 0.8966


In [18]:
import joblib
joblib.dump(mlr,"../models/multiLinearReg.pkl")
joblib.dump(xgboost_model,"../models/xgboost.pkl")
joblib.dump(gboost,"../models/gboost.pkl")
joblib.dump(adaboost,"../models/adaboost.pkl")
joblib.dump(rf_model,"../models/randomForest.pkl")
joblib.dump(catboost_model,"../models/catBoost.pkl")

['../models/catBoost.pkl']